In [1]:
!pip install lifetimes xgboost lightgbm catboost shap mlflow lime --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 118.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.

In [2]:
# CELL 1 — Environment Setup & Logging
#
# UPGRADED: drive.mount() is called FIRST before setting CLV_BASE_DIR.
# Previous version set the env var before mounting — the Drive path did
# not exist yet at that point, which could cause RotatingFileHandler to
# fail silently on the unmounted path.
# =============================================================================
import os, sys, random
import numpy as np
 
# UPGRADED: Mount Drive FIRST — the CLV_BASE_DIR path must exist before
# any module imports try to resolve LOG_FILE or directory paths.
from google.colab import drive
drive.mount('/content/drive')
 
# Now set env var — Drive is mounted, path is accessible
os.environ["CLV_BASE_DIR"] = "/content/drive/MyDrive/clv-prediction-engine"
 
# Global reproducibility — set before any library imports
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
 
# Add project root to path for src.* imports
project_root = os.environ["CLV_BASE_DIR"]
if project_root not in sys.path:
    sys.path.insert(0, project_root)
 
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")
 
# setup_logging() called after drive.mount() — RotatingFileHandler can now
# safely open LOG_FILE on the mounted Drive path.
from src.config import setup_logging
setup_logging()
print("Logging initialised.")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/clv-prediction-engine
Logging initialised.


In [3]:
# =============================================================================
# CELL 2 — Directory Setup & Module Imports (unchanged)
# =============================================================================
from src.config import setup_directories, FEATURE_COLS, SPLIT_DAYS
 
setup_directories()
 
try:
    from src.data_ingestion import load_data, clean_data
    from src.feature_engineering import build_hybrid_features
    from src.modeling import train_and_benchmark, tune_champion_model
    from src.evaluation import evaluate_and_plot, save_model
    print("All modules imported successfully.")
except ModuleNotFoundError as e:
    raise SystemExit(
        f"FATAL: Module import failed — {e}. "
        "Check sys.path and src/__init__.py."
    )

All modules imported successfully.


In [4]:
# Step 1: Load raw data
raw_df   = load_data()

# Step 2: Clean data
clean_df = clean_data(raw_df)

# Step 3: Feature Engineering + Log1p Transform
X_train, X_test, y_train, y_test, y_test_raw = build_hybrid_features(
    clean_df,
    raw_df=raw_df,
    split_days=SPLIT_DAYS,
)

print(f"\ny_train (log-scale) — mean: {y_train.mean():.3f}, max: {y_train.max():.3f}")
print(f"y_test  (log-scale) — mean: {y_test.mean():.3f},  max: {y_test.max():.3f}")
print(f"y_test_raw (dollars) — mean: ${y_test_raw.mean():,.2f}, max: ${y_test_raw.max():,.2f}")

# Segment thresholds from train distribution (stable across runs)
import numpy as np
y_train_raw_recovered = np.expm1(y_train.values)
positive_train        = y_train_raw_recovered[y_train_raw_recovered > 0]
segment_thresholds    = (
    float(np.percentile(positive_train, 20)),
    float(np.percentile(positive_train, 80)),
)
print(f"Segment thresholds (train-derived) — p20: ${segment_thresholds[0]:,.2f} | p80: ${segment_thresholds[1]:,.2f}")

# Steps 4 & 5: Benchmark + Tune
best_model, champ_name, leaderboard = train_and_benchmark(
    X_train, X_test, y_train, y_test
)
tuned_model = tune_champion_model(
    best_model, champ_name,
    X_train, y_train,
    X_test, y_test,
)

# Import CHURN_THRESHOLD so calibration plot is annotated correctly
from src.modeling import CHURN_THRESHOLD

# Steps 6 & 7: Evaluate + Save
evaluate_and_plot(
    tuned_model, champ_name, X_test, y_test, y_test_raw,
    segment_thresholds=segment_thresholds,
    churn_threshold=CHURN_THRESHOLD,
)
save_model(tuned_model, FEATURE_COLS)

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
2026/05/04 03:55:04 INFO mlflow.tracking.fluent: Experiment with name 'CLV_Pipeline_v2.5.0' does not exist. Creating a new experiment.



y_train (log-scale) — mean: 3.667, max: 12.035
y_test  (log-scale) — mean: 3.556,  max: 10.193
y_test_raw (dollars) — mean: $690.22, max: $26,720.70
Segment thresholds (train-derived) — p20: $236.64 | p80: $1,373.14

   MASTER MODEL LEADERBOARD v2.5.0 — Ranked by CV MAE (log-scale)
   Primary metrics: Log-scale | Business metrics: Dollar-scale
   * = Selection ineligible  † = No independent CV (test-set evaluation only)
                      Model  Log_MAE  Log_R2  Dollar_MAE  Dollar_R2     WAPE  CV_MAE_Mean  CV_MAE_Std
       Two-Stage (CatBoost)   2.1978 -0.0948    488.0256     0.5723  70.7057       2.1715      0.1190
  Two-Stage (Random Forest)   2.1309 -0.1072    484.0995     0.5777  70.1369       2.2035      0.0644
        Two-Stage (XGBoost)   2.3603 -0.1806    538.2020     0.4360  77.9753       2.2637      0.1018
       Two-Stage (LightGBM)   2.4426 -0.2471    575.6019     0.3230  83.3939       2.4398      0.0795
        † Weighted Ensemble   2.2140  0.1519    479.9249     0.56

In [5]:
# =============================================================================
# CELL 5 — MLflow Champion Logging  [NEW v2.6.0]
# Logs the final tuned champion model to MLflow Model Registry.
# Run: mlflow ui --port 5000  to view experiment results.
# =============================================================================
from src.modeling import log_champion_to_mlflow, MLFLOW_EXPERIMENT
from src.config import FEATURE_COLS
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# Re-compute final test metrics for champion logging
from src.modeling import LOG_PRED_MAX
_log_preds     = np.clip(tuned_model.predict(X_test), 0, LOG_PRED_MAX)
_dollar_preds  = np.expm1(_log_preds)
_dollar_actual = np.expm1(y_test.values)

_final_log_r2     = r2_score(y_test, _log_preds)
_final_dollar_r2  = r2_score(_dollar_actual, _dollar_preds)
_final_dollar_mae = mean_absolute_error(_dollar_actual, _dollar_preds)

log_champion_to_mlflow(
    tuned_model   = tuned_model,
    champion_name = champ_name,
    dollar_r2     = _final_dollar_r2,
    dollar_mae    = _final_dollar_mae,
    log_r2        = _final_log_r2,
    feature_names = list(FEATURE_COLS),
)

print(f"✅ MLflow champion run logged — experiment: {MLFLOW_EXPERIMENT}")
print("   Launch UI with: mlflow ui --port 5000")
print(f"   Champion: {champ_name} | Dollar R²: {_final_dollar_r2:.4f} | Dollar MAE: ${_final_dollar_mae:,.2f}")


2026/05/04 04:02:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 04:02:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ MLflow champion run logged — experiment: CLV_Pipeline_v2.5.0
   Launch UI with: mlflow ui --port 5000
   Champion: Two-Stage (CatBoost) | Dollar R²: 0.5814 | Dollar MAE: $480.58


Successfully registered model 'CLV_Champion'.
Created version '1' of model 'CLV_Champion'.


In [6]:
import pandas as pd
from pathlib import Path
from src.config import GRAPHS_DIR
 
print("\nPipeline Execution Complete.")
print(f"Champion Model: {champ_name}\n")
 
if leaderboard.empty:
    print("WARNING: Leaderboard is empty — all models failed.")
else:
    # Primary leaderboard — ranked by CV MAE (log-scale)
    print("=== PRIMARY LEADERBOARD (log-scale — model optimisation view) ===")
    display(
        leaderboard[[
            'Model', 'Log_MAE', 'Log_R2',
            'CV_MAE_Mean', 'CV_MAE_Std'
        ]]
        .head(12)
        .style.format({
            'Log_MAE':     '{:.4f}',
            'Log_R2':      '{:.4f}',
            'CV_MAE_Mean': '{:.4f}',
            'CV_MAE_Std':  '{:.4f}',
        })
        .highlight_min(subset=['CV_MAE_Mean'], color='lightgreen')
        .highlight_max(subset=['Log_R2'],      color='lightyellow')
    )
 
    print("\n=== BUSINESS LEADERBOARD (dollar-scale — recruiter-facing view) ===")
    display(
        leaderboard[[
            'Model', 'Dollar_MAE', 'Dollar_R2', 'WAPE', 'SMAPE'
        ]]
        .head(12)
        .style.format({
            'Dollar_MAE': '${:,.2f}',
            'Dollar_R2':  '{:.4f}',
            'WAPE':       '{:.2f}%',
            'SMAPE':      '{:.2f}%',
        })
        .highlight_min(subset=['Dollar_MAE'], color='lightgreen')
        .highlight_max(subset=['Dollar_R2'],  color='lightyellow')
    )
 
# Segment metrics
seg_path = GRAPHS_DIR / 'segment_metrics.csv'
if seg_path.exists():
    seg_df = pd.read_csv(seg_path)
    print("\n=== SEGMENT-LEVEL PERFORMANCE ===")
    display(
        seg_df.style.format({
            'RMSE':       '${:,.2f}',
            'MAE':        '${:,.2f}',
            'R2':         '{:.4f}',
            'WAPE%':      '{:.2f}%',
            'SMAPE%':     '{:.2f}%',
            'Avg_Actual': '${:,.2f}',
            'Avg_Pred':   '${:,.2f}',
        })
        .highlight_max(subset=['R2'],    color='lightyellow')
        .highlight_min(subset=['WAPE%'], color='lightgreen')
    )
 


Pipeline Execution Complete.
Champion Model: Two-Stage (CatBoost)

=== PRIMARY LEADERBOARD (log-scale — model optimisation view) ===


,Model,Log_MAE,Log_R2,CV_MAE_Mean,CV_MAE_Std
0,Two-Stage (CatBoost),2.1978,-0.0948,2.1715,0.1190
1,Two-Stage (Random Forest),2.1309,-0.1072,2.2035,0.0644
2,Two-Stage (XGBoost),2.3603,-0.1806,2.2637,0.1018
3,Two-Stage (LightGBM),2.4426,-0.2471,2.4398,0.0795
4,Weighted Ensemble,2.2140,0.1519,2.4571,0.0000
5,BTYD Statistical Baseline,2.7549,-0.1414,2.6559,1.3932
6,XGBoost,2.4202,0.2664,2.9722,0.1981
7,Ridge (L2),2.5667,0.2503,3.0127,0.3136
8,Random Forest,2.4656,0.2780,3.0203,0.3124
9,Linear Regression,2.5703,0.2487,3.0233,0.3362



=== BUSINESS LEADERBOARD (dollar-scale — recruiter-facing view) ===


,Model,Dollar_MAE,Dollar_R2,WAPE,SMAPE
0,Two-Stage (CatBoost),$488.03,0.5723,70.71%,88.91%
1,Two-Stage (Random Forest),$484.10,0.5777,70.14%,82.48%
2,Two-Stage (XGBoost),$538.20,0.4360,77.98%,95.11%
3,Two-Stage (LightGBM),$575.60,0.3230,83.39%,96.41%
4,Weighted Ensemble,$479.92,0.5698,69.53%,153.46%
5,BTYD Statistical Baseline,$515.40,0.4786,74.67%,130.84%
6,XGBoost,$533.36,0.3254,77.27%,161.30%
7,Ridge (L2),$987.62,-18.9111,143.09%,163.89%
8,Random Forest,$507.80,0.5132,73.57%,163.49%
9,Linear Regression,$991.88,-18.9292,143.71%,163.87%



=== SEGMENT-LEVEL PERFORMANCE ===


,Segment,N,RMSE,MAE,R2,WAPE%,SMAPE%,Avg_Actual,Avg_Pred
0,Top 20% Spenders,76,"$3,562.41","$2,220.96",0.3762,55.43%,85.82%,"$4,006.50","$2,552.79"
1,Mid Spenders,236,$587.21,$431.48,-2.2960,66.66%,96.33%,$647.30,$467.13
2,Low Spenders,61,$184.46,$135.41,-10.3721,95.58%,112.88%,$141.66,$166.75
3,Zero-Spend (Churned),302,$302.25,$150.70,0.0000,nan%,79.47%,$0.00,$150.70
4,All Customers,675,"$1,262.30",$480.58,0.5814,69.63%,89.10%,$690.22,$533.24


py -3.11 -m streamlit run streamlit_app.py


py -3.11 -m mlflow ui --port 5000 

http://127.0.0.1:5000
